# E1 · W3 · Kaggle run 04 — train baseline A (GPU)
**Mục đích:** train YOLOv8n theo `configs/baseline_A.yaml` trên `e1-yolo-v1`, đánh giá `best.pt` trên **val** (test bị khóa), lưu learning curves, PR curve, confusion matrix, metrics theo lớp, SHA-256 weights (issue #18).

**Trước khi chạy:**
1. *Add Input*: `e1-w3-tools` (bản có config An đã duyệt) **và** output của notebook `e1-w3-run03b` — bản có `e1-yolo-v1.zip` (*Add Input → Your Work → Notebooks*). Notebook giải nén vào `/tmp` và kiểm tra SHA-256 từng ảnh với `yolo_manifest.csv.gz` trước khi train.
2. *Settings*: Accelerator **GPU T4 x2** (hoặc P100), Internet **On**. Kiểm tra hạn mức GPU còn lại trước khi chạy.
3. Điền `COMMIT`. `ULTRALYTICS_VERSION` để trống ở lần chạy chính thức đầu tiên (sau đó ghi bản đã dùng vào config).
4. Ở chế độ Edit, chạy thử 4 ô code đầu (ô dữ liệu phải in `images 6869`), rồi mới *Save Version*.

**Sau khi chạy:** *Save Version → Save & Run All (Commit)* (chạy nền). Tải `w3_train_evidence.zip` → `results/W03/kaggle_run_04/`. Tải `runs/baseline_A/weights/best.pt` và `last.pt` → Google Drive `E1/weights/W03/` (không đưa vào Git), gửi link cho Claude.

In [ ]:
COMMIT = ""                 # <-- SHA day du 40 ky tu
ULTRALYTICS_VERSION = ""    # de trong = ban moi nhat; dien vd "8.4.176" de chay lai dung ban

In [ ]:
import glob, hashlib, json, os, sys, zipfile
from pathlib import Path

def find(name):
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    assert hits, f"KHONG TIM THAY {name} trong /kaggle/input — kiem tra Add Input"
    if len(hits) > 1:
        print(f"[canh bao] {name} co {len(hits)} ban, dung ban dau: {hits}")
    return hits[0]

def sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for c in iter(lambda: f.read(1 << 20), b""):
            h.update(c)
    return h.hexdigest()

assert len(COMMIT) == 40 and all(c in "0123456789abcdef" for c in COMMIT.lower()), \
    "Dien COMMIT = SHA day du 40 ky tu cua commit tren main chua cac script (GitHub Desktop -> History -> Copy SHA)"

In [ ]:
!pip -q install {"ultralytics==" + ULTRALYTICS_VERSION if ULTRALYTICS_VERSION else "ultralytics"}
import torch, ultralytics
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "Chua bat GPU: Settings -> Accelerator -> GPU"
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Du lieu YOLO: uu tien e1-yolo-v1.zip (output run 03b dong goi); giai nen vao /tmp de khong lam phinh output notebook
import csv, gzip
EXPECTED_FP = "d0ac7e230b385132c4bbec90eb5af2c5740a478fec6c79aeb245acca032c27de"   # fingerprint run 03b ban dau
zips = sorted(glob.glob("/kaggle/input/**/e1-yolo-v1.zip", recursive=True))
if zips:
    zinfo = json.load(open(find("dataset_zip.json")))
    assert sha(zips[0]) == zinfo["sha256"], "SHA-256 cua e1-yolo-v1.zip khac dataset_zip.json"
    if not os.path.exists("/tmp/e1data/e1-yolo-v1/data.yaml"):
        zipfile.ZipFile(zips[0]).extractall("/tmp/e1data")
    DATA_DIR = "/tmp/e1data/e1-yolo-v1"
else:
    cards = [p for p in glob.glob("/kaggle/input/**/dataset_card.json", recursive=True)
             if os.path.exists(os.path.join(os.path.dirname(p), "data.yaml"))]
    assert len(cards) == 1, f"Can 1 bo du lieu YOLO (e1-yolo-v1.zip hoac thu muc), thay {cards}"
    DATA_DIR = os.path.dirname(cards[0])
card = json.load(open(f"{DATA_DIR}/dataset_card.json"))
MANIFEST = find("yolo_manifest.csv.gz")
man = list(csv.DictReader(gzip.open(MANIFEST, "rt", encoding="utf-8")))
bad = [m["image"] for m in man if not os.path.exists(f"{DATA_DIR}/{m['image']}") or sha(f"{DATA_DIR}/{m['image']}") != m["sha256"]]
assert not bad, f"{len(bad)} anh khac SHA-256 trong manifest, vd {bad[:3]}"
fp = hashlib.sha256("\n".join(f"{m['image']} {m['sha256']}" for m in sorted(man, key=lambda r: r["image"])).encode()).hexdigest()
assert fp == card["content_fingerprint_sha256"], "fingerprint tinh lai khac dataset_card.json"
print("DATA_DIR =", DATA_DIR, "| images", len(man), "| fingerprint", fp[:16], "| trung ban goc:", fp == EXPECTED_FP)
TOOLS = {n: find(n) for n in ["w3_train_baseline.py", "baseline_A.yaml"]}
for n, p in TOOLS.items():
    print(f"{n:24s} sha256={sha(p)[:16]}  {p}")
print(open(TOOLS["baseline_A.yaml"]).read())

In [ ]:
!{sys.executable} {TOOLS["w3_train_baseline.py"]} --data-dir {DATA_DIR} --config {TOOLS["baseline_A.yaml"]} --out /kaggle/working --commit {COMMIT}

## Tự kiểm tra (phải in `PASS`)

In [ ]:
import pandas as pd
ev = "/kaggle/working/w3_train_evidence"
st = json.load(open(f"{ev}/stage_status.json")); print({k: v["status"] for k, v in st.items()})
env = json.load(open(f"{ev}/env.json")); print("GPU:", env.get("gpu"), "| ultralytics", env.get("ultralytics"))
res = pd.read_csv(f"{ev}/train/results.csv"); print("so epoch da chay:", len(res))
m = json.load(open(f"{ev}/baseline_A_val_metrics.json")); print(json.dumps(m["overall"], indent=1))
print(pd.DataFrame(m["per_class"]).to_string(index=False))
ok = all(v["status"] == "OK" for v in st.values()) and env.get("gpu") not in (None, "none") and len(res) > 0
ok &= bool(glob.glob(f"{ev}/val/*PR_curve*.png"))
print("PASS" if ok else "FAIL — gui stage_status.json cho Claude")

In [ ]:
print(open(f"{ev}/weights_manifest.json").read())
p = "/kaggle/working/w3_train_evidence.zip"; print(os.path.getsize(p) // 1024, "KB  sha256", sha(p))